# OPRB: đo thời gian suy luận của cả 5 model trong một phiên

Cả 5 model được đo **lần lượt trong cùng một phiên Kaggle T4**, để mọi số trong bảng hiệu năng đến từ cùng một loại máy.

1. Kaggle: bật **Internet**, Accelerator **GPU T4 x2**.
2. **Save Version → Save & Run All**. Kết quả nằm ở tab Output: `results/<model>/latency.json` và `results/latency_sample100.csv`.
3. Phiên bị ngắt: **Add Input → Notebook Output** của phiên đó rồi chạy lại; model đã đo xong được bỏ qua.

Cách đo (`python -m oprb_eval latency`):

- Cùng 100 ảnh cho mọi model (`--sample 100`, phần đầu của mẫu 3.000 ảnh), batch 1, một tiến trình, 3 ảnh khởi động không tính giờ.
- Mỗi model chạy trong một tiến trình riêng, **nối tiếp nhau**, không có gì chạy song song.
- docTR và hai VLM dùng một GPU (`CUDA_VISIBLE_DEVICES=0`). Tesseract và PP-OCR chạy trên CPU của chính phiên này, không thấy GPU.
- Trọng số được tải trước bằng `pilot --images 1`, nên `load_seconds` chỉ là thời gian nạp, không có thời gian tải.
- Ghi lại: trung bình, trung vị, p95 giây/ảnh, số ảnh lỗi, đỉnh VRAM, số tham số, dung lượng trọng số, CPU và GPU.

Thời gian ước lượng: Tesseract khoảng 20 phút, PP-OCR khoảng 10 phút, docTR vài phút, mỗi VLM khoảng 1–1,5 giờ;
tổng khoảng 3,5–4 giờ GPU kể cả tải dữ liệu và cài thư viện.

In [ ]:
# ===== CẤU HÌNH =====
COMMIT = "main"     # mã commit hoặc tag; ô cuối so fingerprint với results/<model>/run_config.json của các phiên run
SAMPLE = 100        # số ảnh đo, giống nhau cho mọi model
MODELS = ["tesseract", "ppocrv5_mobile_en", "doctr_fast_parseq", "lightonocr2_1b", "qwen3_vl_2b"]
HOURS = 11.0        # không bắt đầu model mới khi đã quá giờ này (phiên tối đa 12 giờ)

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time

START = time.time()
if not pathlib.Path("/kaggle").exists():
    raise SystemExit("Notebook này chạy trên Kaggle, để mọi model được đo trên cùng một loại máy.")
FAST, KEEP = pathlib.Path("/kaggle/temp"), pathlib.Path("/kaggle/working/results")
REPO, DATA = FAST / "oprb-eval", FAST / "oprb_test"
FAST.mkdir(parents=True, exist_ok=True)
KEEP.mkdir(parents=True, exist_ok=True)
os.environ.update(OPRB_CACHE=str(FAST / "cache"), HF_HOME=str(FAST / "hf"), PYTHONUNBUFFERED="1")

def sh(command, **kwargs):
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True, **kwargs)

if not REPO.exists():
    sh(f"git clone -q https://github.com/conthonamanco/oprb-eval.git {REPO}")
sh(f"git fetch -q --tags origin && git checkout -q {COMMIT}", cwd=REPO)
if COMMIT == "main":
    sh("git merge -q --ff-only origin/main", cwd=REPO)
sh("git log -1 --oneline", cwd=REPO)
GPUS = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines()
print("GPU:", GPUS or "không có", "| CPU:", os.cpu_count())
if not GPUS:
    raise SystemExit("Cần GPU T4: đổi Accelerator rồi chạy lại.")

In [ ]:
EXTRA = {"tesseract": "tesseract", "ppocrv5_mobile_en": "ppocr", "doctr_fast_parseq": "doctr",
         "lightonocr2_1b": "vlm", "qwen3_vl_2b": "vlm"}
CPU_ONLY = {"tesseract", "ppocrv5_mobile_en"}

def torch_version():
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True)
    return r.stdout.strip()

TORCH = torch_version()

def install(extra):
    # Mỗi model chỉ cài phần của nó, ngay trước khi đo; pip không được thay torch của máy
    extras = ",".join(dict.fromkeys(["data", extra]))
    sh(f'{sys.executable} -m pip install -q -e "{REPO}[{extras}]"')
    if TORCH and torch_version() != TORCH:
        raise SystemExit("pip đã thay torch của máy. Dừng lại và ghim phiên bản thư viện.")

install("data")
sh(f"{sys.executable} -m unittest discover -s tests", cwd=REPO)

# Khôi phục latency.json của phiên trước (Add Input -> Notebook Output)
for old in sorted(pathlib.Path("/kaggle/input").glob("**/results/*/latency.json")):
    model = old.parent.name
    if model in MODELS and not (KEEP / model / "latency.json").exists():
        (KEEP / model).mkdir(parents=True, exist_ok=True)
        for name in ("latency.json", "run_config.json"):
            if (old.parent / name).exists():
                shutil.copy(old.parent / name, KEEP / model / name)
        print("khôi phục", old)

PY = f"{sys.executable} -m oprb_eval"
sh(f"{PY} download --data {DATA}", cwd=REPO)

In [ ]:
for model in MODELS:
    if (KEEP / model / "latency.json").exists():
        print(f"== {model}: đã đo, bỏ qua")
        continue
    if time.time() - START > HOURS * 3600:
        print(f"== {model}: hết giờ, chưa đo. Chạy lại notebook với Output của phiên này làm Input.")
        continue
    print(f"== {model}", flush=True)
    install(EXTRA[model])
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="" if model in CPU_ONLY else "0")
    # Tải trọng số trước, ra thư mục tạm, để thời gian đo không gồm thời gian tải
    sh(f"{PY} pilot --model {model} --data {DATA} --results {FAST / 'warm'} --images 1", cwd=REPO, env=env)
    sh(f"{PY} latency --model {model} --data {DATA} --results {KEEP} --sample {SAMPLE}", cwd=REPO, env=env)
    print(f"   xong sau {(time.time() - START) / 3600:.2f} giờ", flush=True)

In [ ]:
import csv

rows = []
for model in MODELS:
    path = KEEP / model / "latency.json"
    if not path.exists():
        print("thiếu", model)
        continue
    r = json.loads(path.read_text())
    info, environment = r["model_info"], r["environment"]
    # Thời gian chỉ có nghĩa nếu đo đúng cấu hình đã dùng cho 3.000 ảnh
    committed = json.loads((REPO / "results" / model / "run_config.json").read_text())["fingerprint"]
    if r["fingerprint"] != committed:
        raise SystemExit(f"{model}: cấu hình khác lúc chạy 3.000 ảnh ({r['fingerprint']} != {committed}).")
    rows.append({
        "model": model, "device": info.get("device"), "n": r["n"], "n_failed": r["n_failed"],
        "mean_s": r["mean"], "median_s": r["median"], "p95_s": r["p95"], "load_s": r["load_seconds"],
        "peak_vram_mb": r["peak_vram_mb"], "parameters": info.get("parameters"), "weights_mb": info.get("weights_mb"),
        "cpu": environment.get("cpu"), "cpu_count": environment.get("cpu_count"),
        "gpu": (environment.get("gpus") or [None])[0], "commit": environment.get("commit"),
    })
if rows:
    with open(KEEP / f"latency_sample{SAMPLE}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    for r in rows:
        print(f"{r['model']:20} {r['device']:4} n={r['n']} failed={r['n_failed']} mean={r['mean_s']:.2f}s "
              f"median={r['median_s']:.2f}s p95={r['p95_s']:.2f}s vram={r['peak_vram_mb']} MB")
print(f"Tổng thời gian: {(time.time() - START) / 3600:.2f} giờ")